# Inverse-density sample: even across AoU's premade PC space (replaces both ancestry rounds)

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

## Functions

In [ ]:
import ast

def parse_pcs(col, n=16):
    arr = np.vstack(col.map(ast.literal_eval).to_numpy())
    assert arr.shape[1] == n, arr.shape
    return arr

def hexbin_safe(ax, x, y, **kw):
    """Participant density; cells with fewer than 21 people are not drawn."""
    return ax.hexbin(x, y, **{"gridsize": 120, "bins": "log", "cmap": "Greys", **kw, "mincnt": 21})

def even_weights(Z, k):
    """Inverse kNN density: w ~ r_k^d, so the sample is even over the occupied PC space."""
    from sklearn.neighbors import NearestNeighbors
    r = NearestNeighbors(n_neighbors=k + 1).fit(Z).kneighbors(Z)[0][:, -1]
    return np.maximum(r, r[r > 0].min()) ** Z.shape[1]

def hex_cv(x, y, extent, gridsize=40):
    """Spread of per-hexagon counts over occupied hexagons; lower is more even."""
    h = plt.hexbin(x, y, gridsize=gridsize, extent=extent, mincnt=1); c = h.get_array(); plt.close()
    return float(np.std(c) / np.mean(c))

## Sample

In [ ]:
STAGE = "ancestry_round1"
require_ready(cfg, STAGE)
rep, OUT1, OUT2 = Report(cfg, STAGE), single_target(cfg, STAGE), single_target(cfg, "ancestry_round2")
N, SEED, D, K = cfg.param("uniform_n"), cfg.param("seed"), cfg.param("even_n_pcs"), cfg.param("even_k")

aou = pd.read_csv(cfg.input("ancestry_preds"), sep="\t", usecols=["research_id", "ancestry_pred", "pca_features"],
                  dtype={"research_id": str})
Z = parse_pcs(aou.pop("pca_features"))[:, :D]
w = even_weights(Z, K)
rng = np.random.default_rng(SEED)
pick = np.sort(rng.choice(len(aou), size=N, replace=False, p=w / w.sum()))
rand = np.sort(rng.choice(len(aou), size=N, replace=False))
keep = aou.iloc[pick]

rep.note(uniform_n=N, seed=SEED, even_n_pcs=D, even_k=K)
rep.count("n_participants", len(aou))
print("sampled", rep.count("n_kept", len(keep)))
rep.check("sample size as asked", len(keep) == N, f"{len(keep)}")
rep.check("no duplicate IDs", not keep["research_id"].duplicated().any())

In [ ]:
ext = (Z[:, 0].min(), Z[:, 0].max(), Z[:, 1].min(), Z[:, 1].max())
cv = {"inverse_density": hex_cv(Z[pick, 0], Z[pick, 1], ext), "random": hex_cv(Z[rand, 0], Z[rand, 1], ext)}
rep.note(hex_cv_even=round(cv["inverse_density"], 2), hex_cv_random=round(cv["random"], 2))
rep.check("more even than a random sample (PC1-PC2 hexagon counts)", cv["inverse_density"] < cv["random"],
          f"CV {cv['inverse_density']:.2f} vs {cv['random']:.2f}")

comp = pd.DataFrame({"cohort": aou["ancestry_pred"].value_counts(), "sample": keep["ancestry_pred"].value_counts(),
                     "random_sample": aou.iloc[rand]["ancestry_pred"].value_counts()}).fillna(0).astype(int)
rep.table(comp.reset_index(names="ancestry_pred"), "composition", counts=["cohort", "sample", "random_sample"])

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for row, (i, j) in zip(axes, [(0, 1), (2, 3)]):
    for ax, (lab, ix) in zip(row, [("cohort", slice(None)), ("random sample", rand), ("inverse-density sample", pick)]):
        e = (Z[:, i].min(), Z[:, i].max(), Z[:, j].min(), Z[:, j].max())
        hexbin_safe(ax, Z[ix, i], Z[ix, j], gridsize=15, bins=None, extent=e)
        ax.set(title=lab, xlabel=f"PC{i+1}", ylabel=f"PC{j+1}")
rep.fig(fig, "even_vs_random", "AoU premade PCs; coarse hexagons, linear scale, only those with 21+ people"); plt.show()

## Write (round 1 and round 2 are the same list: no ancestry gate)

In [ ]:
for stage, uri in [(STAGE, OUT1), ("ancestry_round2", OUT2)]:
    if needs_run(uri, FORCE):
        with record(cfg, stage, uri) as meta:
            get_store(uri).write_text(uri, "\n".join(keep["research_id"]) + "\n")
            meta.update(n_kept=len(keep), uniform_n=N, seed=SEED, even_n_pcs=D, even_k=K)
rep.save()